# L11 – Lösningar

In [ ]:
import sys
sys.path.insert(0, "..")
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from dlkit.seed import set_seed
from dlkit.train import fit

device = "cuda" if torch.cuda.is_available() else "cpu"
DATA_DIR = "../data"
loss_fn = nn.CrossEntropyLoss()

def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

weights = models.ResNet18_Weights.DEFAULT
MEAN, STD = weights.transforms().mean, weights.transforms().std
train_tf = transforms.Compose([transforms.RandomResizedCrop(224, scale=(0.7, 1.0)), transforms.RandomHorizontalFlip(),
                               transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224),
                              transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
n = len(datasets.OxfordIIITPet(DATA_DIR, split="trainval", download=True))
perm = np.random.default_rng(42).permutation(n)
train_idx, val_idx = perm[:int(0.8 * n)], perm[int(0.8 * n):]
train_loader = DataLoader(Subset(datasets.OxfordIIITPet(DATA_DIR, split="trainval", transform=train_tf), train_idx),
                          batch_size=64, shuffle=True, num_workers=4)
val_loader = DataLoader(Subset(datasets.OxfordIIITPet(DATA_DIR, split="trainval", transform=eval_tf), val_idx),
                        batch_size=128, num_workers=4)

### Övning 1
Bara det nya huvudet är tränbart: 512 · 37 vikter + 37 bias = **18 981**.

In [ ]:
my_count = 512 * 37 + 37
m = models.resnet18(weights=None)
for p in m.parameters():
    p.requires_grad = False
m.fc = nn.Linear(512, 37)
assert my_count == count_trainable(m)
print("Rätt!", my_count)

### Övning 2

In [ ]:
set_seed(42)
model = models.resnet18(weights=weights)
for p in model.parameters():
    p.requires_grad = False
model.fc = nn.Linear(512, 37)
model = model.to(device)
fit(model, train_loader, val_loader, loss_fn, torch.optim.Adam(model.fc.parameters(), lr=1e-3),
    device, epochs=3, verbose=False)                 # 1. huvudet först

for p in model.layer4.parameters():                 # 2. lås upp det sista blocket
    p.requires_grad = True
print(f"tränbara parametrar: {count_trainable(model):,}")
trainable = [p for p in model.parameters() if p.requires_grad]
h_layer4 = fit(model, train_loader, val_loader, loss_fn,
               torch.optim.AdamW(trainable, lr=1e-4, weight_decay=0.01), device, epochs=4)
assert max(h_layer4["val_acc"]) > 0.85
print("Rätt!", max(h_layer4["val_acc"]))

Att bara låsa upp `layer4` ger oftast nästan samma resultat som full fine-tuning. Det tränar snabbare och har mindre risk för överanpassning, eftersom färre parametrar ändras. Det är ett bra val när datamängden är liten.

### Övning 3

In [ ]:
mn_weights = models.MobileNet_V3_Small_Weights.DEFAULT
mn_tf = mn_weights.transforms()
mn_train_tf = transforms.Compose([transforms.RandomResizedCrop(224, scale=(0.7, 1.0)), transforms.RandomHorizontalFlip(),
                                  transforms.ToTensor(), transforms.Normalize(mn_tf.mean, mn_tf.std)])
mn_eval_tf = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224),
                                 transforms.ToTensor(), transforms.Normalize(mn_tf.mean, mn_tf.std)])
mn_train = DataLoader(Subset(datasets.OxfordIIITPet(DATA_DIR, split="trainval", transform=mn_train_tf), train_idx),
                      batch_size=64, shuffle=True, num_workers=4)
mn_val = DataLoader(Subset(datasets.OxfordIIITPet(DATA_DIR, split="trainval", transform=mn_eval_tf), val_idx),
                    batch_size=128, num_workers=4)

set_seed(42)
mn = models.mobilenet_v3_small(weights=mn_weights)
for p in mn.parameters():
    p.requires_grad = False
mn.classifier[3] = nn.Linear(mn.classifier[3].in_features, 37)
mn = mn.to(device)
print(f"MobileNetV3-small: {sum(p.numel() for p in mn.parameters()):,} parametrar totalt")
t0 = time.time()
h_mn = fit(mn, mn_train, mn_val, loss_fn, torch.optim.Adam(mn.classifier[3].parameters(), lr=1e-3),
           device, epochs=5)
print(f"bästa val acc {max(h_mn['val_acc']):.3f}, {(time.time() - t0) / 5:.1f} s/epok")

MobileNetV3-small har cirka 1,5 miljoner parametrar, jämfört med cirka 11,7 miljoner för ResNet18. Den är mycket snabbare vid prediktion men oftast några procentenheter sämre. Här är tiden per epok ungefär densamma, eftersom inläsningen av bilder är flaskhalsen och inte GPU:n. **Välj MobileNet** när modellen ska köras på mobil, på CPU eller med krav på låg fördröjning. **Välj ResNet eller större modeller** när accuracy är viktigast och hårdvaran räcker.